# 01 — Download ACS PUMS Data for Washington State

This notebook downloads American Community Survey (ACS) Public Use Microdata Sample (PUMS)
person-level files for Washington state from the Census Bureau.

**Data source:** [Census Bureau ACS PUMS](https://www.census.gov/programs-surveys/acs/microdata.html)

**Years:** 2005–2023 (all available ACS 1-Year PUMS), including 2020 experimental estimates

**PUMA vintages:**
- 2000-based PUMAs (used in 2005–2011 ACS)
- 2010-based PUMAs (used in 2012–2021 ACS)
- 2020-based PUMAs (used in 2022+ ACS)

**Note on 2020:** The Census Bureau released 2020 ACS 1-Year data as
*experimental estimates* due to low response rates from COVID-19.
These data have higher uncertainty than other years.

In [1]:
import os
import requests
import zipfile
import io
import pandas as pd
from tqdm.auto import tqdm

DATA_DIR = os.path.abspath(os.path.join('..', 'data'))
RAW_DIR = os.path.join(DATA_DIR, 'raw', 'acs_pums')
DERIVED_DIR = os.path.join(DATA_DIR, 'derived')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DERIVED_DIR, exist_ok=True)

# All available ACS 1-Year PUMS years
YEARS = list(range(2005, 2024))
print(f'Will download ACS 1-Year PUMS for WA state: {YEARS}')


def pums_url(year):
    """Return the Census FTP URL for the WA person PUMS file."""
    if year == 2020:
        return 'https://www2.census.gov/programs-surveys/acs/experimental/2020/data/pums/1-Year/csv_pwa.zip'
    elif year <= 2006:
        return f'https://www2.census.gov/programs-surveys/acs/data/pums/{year}/csv_pwa.zip'
    else:
        return f'https://www2.census.gov/programs-surveys/acs/data/pums/{year}/1-Year/csv_pwa.zip'

Will download ACS 1-Year PUMS for WA state: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


## Download PUMA reference data

The 2010-based PUMA names come from the Census Bureau gazetteer file.
The 2020-based PUMA names are hardcoded from the Census Bureau's
[2020 PUMA Names PDF](https://www2.census.gov/geo/pdfs/reference/puma2020/2020_PUMA_Names.pdf).
The 2000-based PUMAs are identified empirically from the PUMS data.

In [2]:
# Download 2010 PUMA gazetteer (has PUMA codes + names)
gaz_dest = os.path.join(RAW_DIR, '2010_Gaz_PUMAs_national.txt')
if not os.path.exists(gaz_dest):
    url = 'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2010_Gaz_PUMAs_national.zip'
    print(f'Downloading 2010 PUMA gazetteer...')
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        with zf.open('2010_Gaz_PUMAs_national.txt') as src:
            with open(gaz_dest, 'wb') as dst:
                dst.write(src.read())
    print(f'  saved → {gaz_dest}')
else:
    print('2010 PUMA gazetteer: cached')

2010 PUMA gazetteer: cached


In [3]:
# Parse 2010 PUMAs for Washington state
gaz = pd.read_csv(gaz_dest, sep='\t', dtype=str, encoding='latin-1')
gaz.columns = gaz.columns.str.strip()
wa_2010 = gaz[gaz['USPS'] == 'WA'].copy()
# GEOID = state_fips (2) + puma_code (5)
wa_2010['puma'] = wa_2010['GEOID'].str[2:].astype(int)
wa_2010['puma_name'] = wa_2010['NAME']

seattle_2010 = wa_2010[wa_2010['puma_name'].str.contains('Seattle', case=False, na=False)]
print('2010-based Seattle PUMAs (used in 2013–2021 ACS):')
for _, row in seattle_2010.iterrows():
    print(f"  {row['puma']:>5d}  {row['puma_name']}")

print()

# 2020-based Seattle PUMAs (from Census 2020 PUMA Names PDF)
seattle_pumas_2020 = {
    23312: 'Seattle City (West Seattle-Industrial)',
    23313: 'Seattle City (Southeast)',
    23314: 'Seattle City (Central)',
    23315: 'Seattle City (Lake Union-Downtown)',
    23316: 'Seattle City (Northwest)',
    23317: 'Seattle City (Northeast)',
    23318: 'Seattle City (North)',
}
print('2020-based Seattle PUMAs (used in 2022+ ACS):')
for code, name in sorted(seattle_pumas_2020.items()):
    print(f'  {code:>5d}  {name}')

2010-based Seattle PUMAs (used in 2013–2021 ACS):
  11605  Seattle City (West)--Duwamish & Beacon Hill PUMA
  11602  Seattle City (Northeast) PUMA
  11601  Seattle City (Northwest) PUMA
  11603  Seattle City (Downtown)--Queen Anne & Magnolia PUMA
  11604  Seattle City (Southeast)--Capitol Hill PUMA

2020-based Seattle PUMAs (used in 2022+ ACS):
  23312  Seattle City (West Seattle-Industrial)
  23313  Seattle City (Southeast)
  23314  Seattle City (Central)
  23315  Seattle City (Lake Union-Downtown)
  23316  Seattle City (Northwest)
  23317  Seattle City (Northeast)
  23318  Seattle City (North)


## Download ACS PUMS person files

Download the person-level PUMS CSV for Washington state from the Census FTP site.
We extract only the columns needed for commute analysis and save as parquet.

In [4]:
# Columns to extract from each year's PUMS file
TARGET_COLS_UPPER = {
    'SERIALNO', 'ST', 'PUMA', 'PUMA00', 'PUMA10', 'PUMA20',
    'POWSP', 'POWPUMA',
    'JWTR', 'JWTRNS', 'JWRIP', 'DRIVESP',
    'JWMNP', 'JWAP', 'JWDP',
    'PWGTP', 'ESR',
}
# Add replicate weights for standard error computation
for i in range(1, 81):
    TARGET_COLS_UPPER.add(f'PWGTP{i}')


def col_filter(col_name):
    """Filter function for usecols — keep only target columns."""
    return col_name.upper() in TARGET_COLS_UPPER


print(f'Will extract up to {len(TARGET_COLS_UPPER)} columns from each file')

Will extract up to 97 columns from each file


In [5]:
for year in tqdm(YEARS, desc='ACS PUMS download'):
    outfile = os.path.join(DERIVED_DIR, f'acs_pums_wa_commute_{year}.parquet')
    if os.path.exists(outfile):
        print(f'{year}: cached')
        continue

    url = pums_url(year)
    print(f'{year}: downloading from {url}...')

    r = requests.get(url, timeout=600)
    r.raise_for_status()

    # Also save the raw zip to data/raw/acs_pums/
    raw_zip = os.path.join(RAW_DIR, f'csv_pwa_{year}.zip')
    with open(raw_zip, 'wb') as f:
        f.write(r.content)

    content = io.BytesIO(r.content)
    with zipfile.ZipFile(content) as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith('.csv')]
        if not csv_names:
            print(f'{year}: ERROR — no CSV found in zip')
            continue

        print(f'{year}: reading {csv_names[0]}...')
        with zf.open(csv_names[0]) as f:
            df = pd.read_csv(f, usecols=col_filter, low_memory=False)

    # Standardize column names to uppercase
    df.columns = df.columns.str.upper()
    df['YEAR'] = year

    df.to_parquet(outfile, index=False)
    print(f'{year}: saved {len(df):,} records')

ACS PUMS download:   0%|          | 0/19 [00:00<?, ?it/s]

2005: downloading from https://www2.census.gov/programs-surveys/acs/data/pums/2005/csv_pwa.zip...


2005: reading ss05pwa.csv...


2005: saved 61,520 records
2006: downloading from https://www2.census.gov/programs-surveys/acs/data/pums/2006/csv_pwa.zip...


2006: reading ss06pwa.csv...


...

2011: saved 69,762 records
2012: downloading from https://www2.census.gov/programs-surveys/acs/data/pums/2012/1-Year/csv_pwa.zip...


2012: reading ss12pwa.csv...


2012: saved 69,301 records
2013: cached
2014: cached
2015: cached
2016: cached
2017: cached
2018: cached
2019: cached
2020: downloading from https://www2.census.gov/programs-surveys/acs/experimental/2020/data/pums/1-Year/csv_pwa.zip...


2020: reading psam_p53.csv...


2020: saved 65,452 records
2021: cached
2022: cached
2023: cached


## Summary

In [6]:
print('Downloaded data summary')
print('=' * 75)
total = 0
for year in YEARS:
    f = os.path.join(DERIVED_DIR, f'acs_pums_wa_commute_{year}.parquet')
    if os.path.exists(f):
        df = pd.read_parquet(f)
        n = len(df)
        total += n
        commute_cols = [c for c in df.columns if c in ('JWTR', 'JWTRNS')]
        rip_cols = [c for c in df.columns if c in ('JWRIP', 'DRIVESP')]
        exp = ' (experimental)' if year == 2020 else ''
        print(f'  {year}{exp}:  {n:>8,} records   mode_var={commute_cols}  rip_var={rip_cols}')
    else:
        print(f'  {year}:  NOT DOWNLOADED')
print(f'{"":>6s}  {total:>8,} total records')

Downloaded data summary
  2005:    61,520 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']
  2006:    63,524 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']
  2007:    64,414 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']


  2008:    65,386 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']
  2009:    66,407 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']
  2010:    67,388 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']
  2011:    69,762 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']
  2012:    69,301 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']


  2013:    69,593 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']


  2014:    70,600 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']


  2015:    71,804 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']


...

  2018:    76,225 records   mode_var=['JWTR']  rip_var=['JWRIP', 'DRIVESP']


  2019:    77,879 records   mode_var=['JWTRNS']  rip_var=['JWRIP', 'DRIVESP']
  2020 (experimental):    65,452 records   mode_var=['JWTRNS']  rip_var=['JWRIP', 'DRIVESP']


  2021:    78,528 records   mode_var=['JWTRNS']  rip_var=['JWRIP', 'DRIVESP']


  2022:    80,818 records   mode_var=['JWTRNS']  rip_var=['JWRIP', 'DRIVESP']


  2023:    81,826 records   mode_var=['JWTRNS']  rip_var=['JWRIP', 'DRIVESP']
        1,347,505 total records
